In [4]:
import pandas as pd
import pymysql
from typing import Optional, List, Tuple, Dict
import warnings
warnings.filterwarnings('ignore')
from DATA.stock_invest_function import get_db_host
from DATA.us_target_ticker_list_2000 import ticker_list

def get_shares_outstanding_timeseries(
    ticker: str,
    user: str,
    password: str,
    host: str,
    port: int,
    database: str,
    item_name: str = 'shroutdi'  # 희석 발행주식수
) -> Optional[pd.DataFrame]:
    """
    특정 티커의 발행주식수 시계열 데이터 추출

    Parameters:
    -----------
    ticker : str
        종목 티커
    item_name : str
        추출할 항목 ('shroutdi': 희석주식수, 'shrout': 기본주식수)

    Returns:
    --------
    pd.DataFrame or None
        인덱스: date (datetime)
        컬럼: shroutdi (or shrout)
    """
    try:
        connection = pymysql.connect(
            user=user,
            password=password,
            host=host,
            port=port,
            database=database,
            charset='utf8mb4'
        )

        query = """
        SELECT date, value
        FROM US_IS_from_FMP
        WHERE ticker = %s
          AND item = %s
        ORDER BY date
        """

        df = pd.read_sql(query, connection, params=(ticker, item_name))
        connection.close()

        if df.empty:
            print(f"Warning: {ticker}에 대한 {item_name} 데이터가 없습니다.")
            return None

        # date를 인덱스로 설정
        df['date'] = pd.to_datetime(df['date'])
        df.set_index('date', inplace=True)

        # 컬럼명 변경
        df.columns = [item_name]

        # 중복 제거 (같은 날짜에 여러 값이 있는 경우 최신값 사용)
        df = df[~df.index.duplicated(keep='last')]

        return df

    except Exception as e:
        print(f"Error extracting data for {ticker}: {str(e)}")
        return None


def get_multiple_shares_outstanding(
    tickers: List[str],
    user: str,
    password: str,
    host: str,
    port: int,
    database: str,
    item_name: str = 'shroutdi'
) -> Tuple[Dict[str, pd.DataFrame], List[str]]:
    """
    여러 티커의 발행주식수 데이터 일괄 추출

    Parameters:
    -----------
    tickers : List[str]
        티커 리스트
    item_name : str
        추출할 항목 ('shroutdi' 또는 'shrout')

    Returns:
    --------
    Tuple[Dict[str, pd.DataFrame], List[str]]
        성공한 데이터 딕셔너리, 실패한 티커 리스트
    """
    results = {}
    errors = []

    for ticker in tickers:
        print(f"Processing {ticker}...", end=' ')

        df = get_shares_outstanding_timeseries(
            ticker=ticker,
            user=user,
            password=password,
            host=host,
            port=port,
            database=database,
            item_name=item_name
        )

        if df is not None:
            results[ticker] = df
            print(f"OK ({len(df)} records)")
        else:
            errors.append(ticker)
            print("FAILED")

    return results, errors


In [5]:
# 데이터베이스 연결 정보
DB_CONFIG = {
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'host': get_db_host(),
    'port': 3307,
    'database': 'investar'
}

# 예시 1: 단일 티커 추출
print("=" * 60)
print("단일 티커 추출 (희석 발행주식수)")
print("=" * 60)

df_hpe = get_shares_outstanding_timeseries(
    ticker='HPE',
    **DB_CONFIG
)

if df_hpe is not None:
    print(f"\nHPE 데이터 요약:")
    print(f"기간: {df_hpe.index.min()} ~ {df_hpe.index.max()}")
    print(f"데이터 포인트: {len(df_hpe)}개")
    print(f"\n최근 5개 데이터:")
    print(df_hpe.tail())
    print(f"\n기초 통계:")
    print(df_hpe.describe())

# 예시 1-2: 기본 발행주식수 추출
print("\n" + "=" * 60)
print("단일 티커 추출 (기본 발행주식수)")
print("=" * 60)

# df_hpe_basic = get_shares_outstanding_timeseries(
#     ticker='HPE',
#     item_name='shrout',  # 기본 발행주식수
#     **DB_CONFIG
# )
#
# if df_hpe_basic is not None:
#     print(f"\nHPE 기본 발행주식수 데이터:")
#     print(df_hpe_basic.tail())
#
# # 예시 2: 여러 티커 일괄 추출
# print("\n" + "=" * 60)
# print("여러 티커 일괄 추출")
# print("=" * 60)

# ticker_list = ['HPE', 'ATEN', 'DIS', 'INVALID_TICKER', 'XOM']

results, error_list = get_multiple_shares_outstanding(
    tickers=ticker_list,
    **DB_CONFIG
)

print(f"\n성공: {len(results)}개 티커")
print(f"실패: {len(error_list)}개 티커")

if error_list:
    print(f"\n오류 티커 목록: {error_list}")

# 성공한 데이터 확인
print("\n추출된 데이터 요약:")
for ticker, df in results.items():
    print(f"\n{ticker}:")
    print(f"  기간: {df.index.min().strftime('%Y-%m-%d')} ~ {df.index.max().strftime('%Y-%m-%d')}")
    print(f"  데이터 포인트: {len(df)}개")
    print(f"  최근 값: {df.iloc[-1]['shroutdi']:,.0f}")

단일 티커 추출 (희석 발행주식수)

HPE 데이터 요약:
기간: 2014-01-31 00:00:00 ~ 2025-12-31 00:00:00
데이터 포인트: 144개

최근 5개 데이터:
                shroutdi
date                    
2025-08-31  1.421000e+09
2025-09-30  1.421000e+09
2025-10-31  1.324000e+09
2025-11-30  1.324000e+09
2025-12-31  1.324000e+09

기초 통계:
           shroutdi
count  1.440000e+02
mean   1.491172e+09
std    2.026689e+08
min    1.287000e+09
25%    1.322750e+09
50%    1.360000e+09
75%    1.702250e+09
max    1.826000e+09

단일 티커 추출 (기본 발행주식수)
Processing NVDA... OK (264 records)
Processing GOOG... OK (262 records)
Processing AAPL... OK (262 records)
Processing MSFT... OK (262 records)
Processing AMZN... OK (262 records)
Processing TSM... OK (262 records)
Processing META... OK (178 records)
Processing AVGO... OK (210 records)
Processing TSLA... OK (226 records)
Processing LLY... OK (262 records)
Processing WMT... OK (264 records)
Processing XOM... OK (262 records)
Processing ASML... OK (262 records)
Processing JNJ... OK (262 records)
Processing O

KeyboardInterrupt: 

In [2]:
from DATA.stock_invest_function import get_db_host

# 데이터베이스 연결 정보
DB_CONFIG = {
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'host': get_db_host(),
    'port': 3307,
    'database': 'investar'
}

# 예시 1: 단일 티커 추출
print("=" * 60)
print("단일 티커 추출")
print("=" * 60)

df_hpe = get_shares_outstanding_timeseries(
    ticker='HPE',
    **DB_CONFIG
)

if df_hpe is not None:
    print(f"\nHPE 데이터 요약:")
    print(f"기간: {df_hpe.index.min()} ~ {df_hpe.index.max()}")
    print(f"데이터 포인트: {len(df_hpe)}개")
    print(f"\n최근 5개 데이터:")
    print(df_hpe.tail())
    print(f"\n기초 통계:")
    print(df_hpe.describe())

# 예시 2: 여러 티커 일괄 추출
print("\n" + "=" * 60)
print("여러 티커 일괄 추출")
print("=" * 60)

ticker_list = ['HPE', 'AAPL', 'MSFT', 'INVALID_TICKER', 'GOOGL']

results, error_list = get_multiple_shares_outstanding(
    tickers=ticker_list,
    **DB_CONFIG
)

print(f"\n성공: {len(results)}개 티커")
print(f"실패: {len(error_list)}개 티커")

if error_list:
    print(f"\n오류 티커 목록: {error_list}")

# 성공한 데이터 확인
print("\n추출된 데이터 요약:")
for ticker, df in results.items():
    print(f"\n{ticker}:")
    print(f"  기간: {df.index.min().strftime('%Y-%m-%d')} ~ {df.index.max().strftime('%Y-%m-%d')}")
    print(f"  데이터 포인트: {len(df)}개")
    print(f"  최근 값: {df.iloc[-1]['shroutdi']:,.0f}")


단일 티커 추출

여러 티커 일괄 추출
✗ HPE: No data found
✗ AAPL: No data found
✗ MSFT: No data found
✗ INVALID_TICKER: No data found
✗ GOOGL: No data found

성공: 0개 티커
실패: 5개 티커

오류 티커 목록: ['HPE', 'AAPL', 'MSFT', 'INVALID_TICKER', 'GOOGL']

추출된 데이터 요약:


In [1]:
import pandas as pd
import pymysql
from typing import Optional, List, Tuple
import warnings
warnings.filterwarnings('ignore')
from DATA.stock_invest_function import get_db_host

# DB 연결 정보
DB_CONFIG = {
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'host': get_db_host(),
    'port': 3307,
    'database': 'investar'
}

# 1. 테이블 구조 확인
print("=" * 60)
print("테이블 구조 확인")
print("=" * 60)

try:
    connection = pymysql.connect(**DB_CONFIG, charset='utf8mb4')

    # 테이블 구조
    query = "DESCRIBE US_IS_from_FMP"
    df_structure = pd.read_sql(query, connection)
    print("\n테이블 구조:")
    print(df_structure)

    # 2. 샘플 데이터 확인
    print("\n" + "=" * 60)
    print("샘플 데이터 확인 (상위 5개)")
    print("=" * 60)

    query = "SELECT * FROM US_IS_from_FMP LIMIT 5"
    df_sample = pd.read_sql(query, connection)
    print(df_sample)

    # 3. 티커 목록 확인
    print("\n" + "=" * 60)
    print("저장된 티커 목록 (상위 10개)")
    print("=" * 60)

    query = "SELECT DISTINCT ticker FROM US_IS_from_FMP LIMIT 10"
    df_tickers = pd.read_sql(query, connection)
    print(df_tickers)

    # 4. item 목록 확인
    print("\n" + "=" * 60)
    print("저장된 item 목록")
    print("=" * 60)

    query = "SELECT DISTINCT item FROM US_IS_from_FMP"
    df_items = pd.read_sql(query, connection)
    print(df_items)

    # 5. weightedAverageShsOutDil 데이터 존재 확인
    print("\n" + "=" * 60)
    print("weightedAverageShsOutDil 데이터 확인")
    print("=" * 60)

    query = """
    SELECT ticker, COUNT(*) as cnt
    FROM US_IS_from_FMP
    WHERE item = 'weightedAverageShsOutDil'
    GROUP BY ticker
    LIMIT 10
    """
    df_check = pd.read_sql(query, connection)
    print(df_check)

    # 6. HPE 티커의 모든 데이터 확인
    print("\n" + "=" * 60)
    print("HPE 티커의 모든 데이터")
    print("=" * 60)

    query = """
    SELECT ticker, item, COUNT(*) as cnt
    FROM US_IS_from_FMP
    WHERE ticker = 'HPE'
    GROUP BY ticker, item
    """
    df_hpe = pd.read_sql(query, connection)
    print(df_hpe)

    connection.close()

except Exception as e:
    print(f"Error: {str(e)}")


테이블 구조 확인

테이블 구조:
         Field      Type Null  Key Default Extra
0         date  datetime  YES         None      
1  report_date  datetime  YES         None      
2       ticker      text  YES  MUL    None      
3       period      text  YES         None      
4   date_month      text  YES         None      
5         item      text  YES         None      
6        value    double  YES         None      
7      fs_name      text  YES         None      

샘플 데이터 확인 (상위 5개)
        date report_date ticker period date_month  item         value fs_name
0 2014-01-31  2014-01-31    HPE     Q1    2014-01  sale  1.367300e+10      is
1 2014-02-28  2014-01-31    HPE     Q1    2014-01  sale  1.367300e+10      is
2 2014-03-31  2014-01-31    HPE     Q1    2014-01  sale  1.367300e+10      is
3 2014-04-30  2014-04-30    HPE     Q2    2014-04  sale  1.367300e+10      is
4 2014-05-31  2014-04-30    HPE     Q2    2014-04  sale  1.367300e+10      is

저장된 티커 목록 (상위 10개)
  ticker
0    HPE
1   ATEN
2    